# Tugas Praktikum 4: Association Rule

## Mengimpor Package

In [4]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import zipfile
import requests
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.frequent_patterns import association_rules

## Mengimpor Dataset

In [5]:
if not os.path.exists("Online Retail.xlsx"):
    dataset_url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    req = requests.get(dataset_url, stream=True)

    with req:
        with open("download.zip", "wb") as f:
            for chunk in req.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    
    if zipfile.is_zipfile("download.zip"):
        with zipfile.ZipFile('download.zip') as z:
            z.extractall('.')
            print('extracted:', z.namelist())
        
df = pd.read_excel("Online Retail.xlsx")

## Initial EDA

In [6]:
print(df.shape)
display(df.info())
display(df.head())
display(df.describe())

(541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


None

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## Praprocess Data

### Menghapus Fitur yang Tidak Relevan

In [7]:
to_drop = [
    col for col in df.columns if col not in ["InvoiceNo", "StockCode", "Description", "Quantity"]
]

df = df.drop(columns=to_drop)
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   InvoiceNo    541909 non-null  object
 1   StockCode    541909 non-null  object
 2   Description  540455 non-null  object
 3   Quantity     541909 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 16.5+ MB


None

### Memperbaiki Tipe Data Tiap Fitur

In [8]:
for col, datatype in zip(
    df.columns, [str, str, str, int]
):
    df[col] = df[col].map(lambda x: x if isinstance(x, float) else datatype(x))
    
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   InvoiceNo    541909 non-null  object
 1   StockCode    541909 non-null  object
 2   Description  540455 non-null  object
 3   Quantity     541909 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 16.5+ MB


None

### Menghapus Baris yang Tidak Relevan atau Tidak Valid

In [9]:
# df = df.dropna()
print("Invoice count (before):", len(df["InvoiceNo"]))

quantity_mask = df["Quantity"] > 0
invoice_mask = df["InvoiceNo"].map(lambda x: not x.startswith("C"))
combined_mask = (quantity_mask & invoice_mask)
df = df[combined_mask].dropna()

print("Invoice count (after):", len(df["InvoiceNo"]))
display(df.info())

Invoice count (before): 541909
Invoice count (after): 530693
<class 'pandas.core.frame.DataFrame'>
Index: 530693 entries, 0 to 541908
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   InvoiceNo    530693 non-null  object
 1   StockCode    530693 non-null  object
 2   Description  530693 non-null  object
 3   Quantity     530693 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 20.2+ MB


None

### Menghapus Fitur Redundan dan Fitur Tidak Relevan

In [10]:
stock_desc_map = {
    stock: desc for stock, desc in zip(df["StockCode"], df["Description"])
}

to_drop = ["Quantity", "Description"]
df = df.drop(columns=to_drop)
display(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 530693 entries, 0 to 541908
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   InvoiceNo  530693 non-null  object
 1   StockCode  530693 non-null  object
dtypes: object(2)
memory usage: 12.1+ MB


None

### Melakukan One-Hot Encoding

In [11]:
basket = (
    pd.crosstab(df["InvoiceNo"], df["StockCode"])
).astype(bool)

display(basket.head())
print(basket.info())

StockCode,10002,10080,10120,10123C,10124A,10124G,10125,10133,10135,11001,...,M,PADS,POST,S,gift_0001_10,gift_0001_20,gift_0001_30,gift_0001_40,gift_0001_50,m
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


<class 'pandas.core.frame.DataFrame'>
Index: 20136 entries, 536365 to A563187
Columns: 3925 entries, 10002 to m
dtypes: bool(3925)
memory usage: 75.5+ MB
None


## Evaluasi Model Association Rule

In [12]:
min_support = 0.01
min_confidence = 0.5

# # Filter item yang benar-benar memenuhi support agar Apriori dan FP-Growth tidak
# # membangun kombinasi yang terlalu besar pada data dengan banyak StockCode unik.
# item_support = basket.sum(axis=0) / len(basket)
# eligible_items = item_support[item_support >= min_support].sort_values(ascending=False)

# print(f"Jumlah item sebelum filter: {basket.shape[1]}")
# print(f"Jumlah item yang memenuhi support >= {min_support}: {len(eligible_items)}")

# # Batasi agar tetap aman secara memori jika dataset masih sangat tinggi kardinalitas item-nya.
# if len(eligible_items) > 2000:
#     eligible_items = eligible_items.head(2000)
#     print(f"Membatasi item yang dianalisis ke top {len(eligible_items)} item paling sering muncul.")

# basket = basket[eligible_items.index]

# 1) Apriori
now = time.time()
apriori_freq = apriori(basket, min_support=min_support, use_colnames=True)
apriori_rules = association_rules(apriori_freq, metric="confidence", min_threshold=min_confidence)
elapsed = time.time() - now
apriori_rules = apriori_rules[["antecedents", "consequents", "support", "confidence", "lift"]]
apriori_rules = apriori_rules.sort_values(
    ["lift", "confidence", "support"],
    ascending=[False, False, False]
).reset_index(drop=True)
apriori_rules["antecedents"] = apriori_rules["antecedents"].map(
    lambda stock: [stock_desc_map[item] for item in tuple(stock)]
)
apriori_rules["consequents"] = apriori_rules["consequents"].map(
    lambda stock: [stock_desc_map[item] for item in tuple(stock)]
)

print(f"=== Hasil Rule Apriori ({elapsed}s) ===")
display(apriori_rules)
print("\nTop 3 Rule Apriori (descending):")
display(apriori_rules.head(3))

# 2) FP-Growth
now = time.time()
fpgrowth_freq = fpgrowth(basket, min_support=min_support, use_colnames=True)
fpgrowth_rules = association_rules(fpgrowth_freq, metric="confidence", min_threshold=min_confidence)
elapsed = time.time() - now
fpgrowth_rules = fpgrowth_rules[["antecedents", "consequents", "support", "confidence", "lift"]]
fpgrowth_rules = fpgrowth_rules.sort_values(
    ["lift", "confidence", "support"],
    ascending=[False, False, False]
).reset_index(drop=True)
fpgrowth_rules["antecedents"] = fpgrowth_rules["antecedents"].map(
    lambda stock: [stock_desc_map[item] for item in tuple(stock)]
)
fpgrowth_rules["consequents"] = fpgrowth_rules["consequents"].map(
    lambda stock: [stock_desc_map[item] for item in tuple(stock)]
)

print(f"\n=== Hasil Rule FP-Growth ({elapsed}s) ===")
display(fpgrowth_rules)
print("\nTop 3 Rule FP-Growth (descending):")
display(fpgrowth_rules.head(3))

=== Hasil Rule Apriori (18.78351926803589s) ===


,antecedents,consequents,support,confidence,lift
0,"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",[HERB MARKER THYME],0.010081,0.944186,80.219959
1,[HERB MARKER THYME],"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",0.010081,0.856540,80.219959
2,[HERB MARKER ROSEMARY],"[HERB MARKER THYME, HERB MARKER PARSLEY]",0.010081,0.845833,79.961033
3,"[HERB MARKER THYME, HERB MARKER PARSLEY]",[HERB MARKER ROSEMARY],0.010081,0.953052,79.961033
4,[HERB MARKER THYME],[HERB MARKER ROSEMARY],0.010975,0.932489,78.235865
...,...,...,...,...,...
932,[PINK REGENCY TEACUP AND SAUCER],[REGENCY CAKESTAND 3 TIER],0.019468,0.511749,5.180787
933,[SUKI SHOULDER BAG],[JUMBO BAG RED RETROSPOT],0.014005,0.532075,5.121354
934,[GREEN REGENCY TEACUP AND SAUCER],[REGENCY CAKESTAND 3 TIER],0.025278,0.501478,5.076801
935,[TOY TIDY PINK POLKADOT],[JUMBO BAG RED RETROSPOT],0.012118,0.520256,5.007587



Top 3 Rule Apriori (descending):


,antecedents,consequents,support,confidence,lift
0,"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",[HERB MARKER THYME],0.010081,0.944186,80.219959
1,[HERB MARKER THYME],"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",0.010081,0.856540,80.219959
2,[HERB MARKER ROSEMARY],"[HERB MARKER THYME, HERB MARKER PARSLEY]",0.010081,0.845833,79.961033



=== Hasil Rule FP-Growth (85.30117225646973s) ===


,antecedents,consequents,support,confidence,lift
0,"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",[HERB MARKER THYME],0.010081,0.944186,80.219959
1,[HERB MARKER THYME],"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",0.010081,0.856540,80.219959
2,[HERB MARKER ROSEMARY],"[HERB MARKER THYME, HERB MARKER PARSLEY]",0.010081,0.845833,79.961033
3,"[HERB MARKER THYME, HERB MARKER PARSLEY]",[HERB MARKER ROSEMARY],0.010081,0.953052,79.961033
4,[HERB MARKER THYME],[HERB MARKER ROSEMARY],0.010975,0.932489,78.235865
...,...,...,...,...,...
932,[PINK REGENCY TEACUP AND SAUCER],[REGENCY CAKESTAND 3 TIER],0.019468,0.511749,5.180787
933,[SUKI SHOULDER BAG],[JUMBO BAG RED RETROSPOT],0.014005,0.532075,5.121354
934,[GREEN REGENCY TEACUP AND SAUCER],[REGENCY CAKESTAND 3 TIER],0.025278,0.501478,5.076801
935,[TOY TIDY PINK POLKADOT],[JUMBO BAG RED RETROSPOT],0.012118,0.520256,5.007587



Top 3 Rule FP-Growth (descending):


,antecedents,consequents,support,confidence,lift
0,"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",[HERB MARKER THYME],0.010081,0.944186,80.219959
1,[HERB MARKER THYME],"[HERB MARKER ROSEMARY, HERB MARKER PARSLEY]",0.010081,0.856540,80.219959
2,[HERB MARKER ROSEMARY],"[HERB MARKER THYME, HERB MARKER PARSLEY]",0.010081,0.845833,79.961033
